In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamp as UTC datetimes (keep original numeric timestamp as well)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Ensure timestamp columns are numeric (they should be Unix seconds) and create datetime columns
for df in (metric_df, trace_df, log_df, error_logs_df):
    # coerce to numeric in case of stray strings, then keep numeric timestamp column
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
    df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) Metric overview: per (cmdb_id, kpi_name) compute count, min_ts, max_ts, p95_value (global)
metric_grp = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
        p95_value=('value', lambda s: s.quantile(0.95))
    )
    .reset_index()
)

# Add readable datetime strings for min/max
metric_grp['min_timestamp_dt'] = pd.to_datetime(metric_grp['min_timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
metric_grp['max_timestamp_dt'] = pd.to_datetime(metric_grp['max_timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Sort by count desc and limit to 50 rows; select/arrange columns as requested
metric_overview = (
    metric_grp
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'min_timestamp_dt', 'max_timestamp', 'max_timestamp_dt', 'p95_value']]
    .head(50)
)

# 2) Trace overview: per (cmdb_id, trace_name) compute count, min_ts, max_ts
trace_grp = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max')
    )
    .reset_index()
)
trace_grp['min_timestamp_dt'] = pd.to_datetime(trace_grp['min_timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
trace_grp['max_timestamp_dt'] = pd.to_datetime(trace_grp['max_timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

trace_overview = (
    trace_grp
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id', 'trace_name', 'count', 'min_timestamp', 'min_timestamp_dt', 'max_timestamp', 'max_timestamp_dt']]
    .head(50)
)

# 3) Log overview: per (cmdb_id, log_name) compute count, min_ts, max_ts
log_grp = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(
        count=('value', 'count'),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max')
    )
    .reset_index()
)
log_grp['min_timestamp_dt'] = pd.to_datetime(log_grp['min_timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_grp['max_timestamp_dt'] = pd.to_datetime(log_grp['max_timestamp'], unit='s', utc=True).dt.strftime('%Y-%m-%dT%H:%M:%SZ')

log_overview = (
    log_grp
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id', 'log_name', 'count', 'min_timestamp', 'min_timestamp_dt', 'max_timestamp', 'max_timestamp_dt']]
    .head(50)
)

# 4) error_logs count and earliest/latest timestamps
if len(error_logs_df) > 0:
    total_error_rows = int(len(error_logs_df))
    earliest_ts = int(error_logs_df['timestamp'].min())
    latest_ts = int(error_logs_df['timestamp'].max())
    earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
    latest_dt = pd.to_datetime(latest_ts, unit='s', utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
else:
    total_error_rows = 0
    earliest_ts = pd.NA
    latest_ts = pd.NA
    earliest_dt = None
    latest_dt = None

error_summary = pd.DataFrame([{
    'total_rows': total_error_rows,
    'earliest_timestamp': earliest_ts,
    'earliest_timestamp_dt': earliest_dt,
    'latest_timestamp': latest_ts,
    'latest_timestamp_dt': latest_dt
}])

# Display the four compact summaries (each limited to at most 50 rows)
metric_overview, trace_overview, log_overview, error_summary

```
Out[1]:
```
```python
# Prepare a plain-English summary string based on the previously computed overviews
total_metric_groups = int(metric_grp.shape[0])
metric_max_count = int(metric_grp['count'].max())
metric_groups_with_max = int((metric_grp['count'] == metric_max_count).sum())
pct_with_max = round(100.0 * metric_groups_with_max / total_metric_groups, 1) if total_metric_groups else 0.0

# Top 5 metric groups by global P95 value (include name, kpi, p95, and count)
top5_p95 = metric_grp.nlargest(5, 'p95_value')[['cmdb_id', 'kpi_name', 'p95_value', 'count']].reset_index(drop=True)
top5_lines = []
for i, row in top5_p95.iterrows():
    top5_lines.append(f"{i+1}. {row['cmdb_id']} / {row['kpi_name']} — P95={row['p95_value']:.6g}, points={int(row['count'])}")

total_trace_groups = int(trace_grp.shape[0])
trace_max_count = int(trace_grp['count'].max())

total_log_groups = int(log_grp.shape[0])
log_max_count = int(log_grp['count'].max())

total_error_rows = int(error_summary['total_rows'].iloc[0])
earliest_error_ts = error_summary['earliest_timestamp'].iloc[0]
earliest_error_dt = error_summary['earliest_timestamp_dt'].iloc[0]
latest_error_ts = error_summary['latest_timestamp'].iloc[0]
latest_error_dt = error_summary['latest_timestamp_dt'].iloc[0]

summary = (
    "Summary of loaded telemetry (plain English):\n"
    f"- Files loaded: metric.csv, trace.csv, log.csv, error_logs.csv.\n"
    f"- Metric overview: {total_metric_groups} distinct (cmdb_id, kpi_name) groups found. "
    f"{metric_groups_with_max} groups ({pct_with_max}%) have the maximum of {metric_max_count} data points (full-series coverage in the observed window).\n"
    f"- Top metric groups by global P95 (top 5):\n  " + "\n  ".join(top5_lines) + "\n"
    f"- Trace overview: {total_trace_groups} distinct (cmdb_id, trace_name) groups; most groups have up to {trace_max_count} points.\n"
    f"- Log overview: {total_log_groups} distinct (cmdb_id, log_name) groups; most groups have up to {log_max_count} points.\n"
    f"- Error logs: total rows = {total_error_rows}."
)

# If there are error log timestamps, append earliest/latest info
if total_error_rows > 0:
    summary += f" Earliest error_log timestamp = {earliest_error_ts} ({earliest_error_dt}), latest = {latest_error_ts} ({latest_error_dt})."
else:
    summary += " No error_log rows were found."

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id kpi_name  count  min_timestamp      min_timestamp_dt  max_timestamp      max_timestamp_dt     p95_value
353           ts-voucher-service   socket     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  4.000000e+00
0    ts-admin-basic-info-service      cpu     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  1.062521e+00
314             ts-travel2-mongo   diskio     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  4.019026e+05
315             ts-travel2-mongo      mem     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  6.684366e+07
316             ts-travel2-mongo   socket     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  5.000000e+00
..                           ...      ...    ...            ...                   ...            ...                   ...           ...
53             ts-avatar-service      mem     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  1.338976e+08
54             ts-avatar-service   socket     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  5.000000e+00
55              ts-basic-service      cpu     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  1.226651e+00
56              ts-basic-service   diskio     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  6.118998e+04
57              ts-basic-service    error     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z  0.000000e+00

[50 rows x 8 columns],                      cmdb_id                                         trace_name  count  min_timestamp      min_timestamp_dt  max_timestamp      max_timestamp_dt
399       ts-travel2-service              trace.to_ts-travel2-service.row_count     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
0                       root  trace.to_ts-admin-basic-info-service.duration_...     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
1                       root  trace.to_ts-admin-basic-info-service.duration_p95     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
2                       root    trace.to_ts-admin-basic-info-service.error_rate     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
3                       root     trace.to_ts-admin-basic-info-service.row_count     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
..                       ...                                                ...    ...            ...                   ...            ...                   ...
341        ts-travel-service                       trace.from_root.duration_p95     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
342        ts-travel-service                         trace.from_root.error_rate     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
343        ts-travel-service                          trace.from_root.row_count     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
72   ts-admin-travel-service     trace.to_ts-admin-travel-service.duration_mean     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
73   ts-admin-travel-service      trace.to_ts-admin-travel-service.duration_p95     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z

[50 rows x 7 columns],                         cmdb_id         log_name  count  min_timestamp      min_timestamp_dt  max_timestamp      max_timestamp_dt
0   ts-admin-basic-info-service  log.error_count     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
1   ts-admin-basic-info-service    log.row_count     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
2       ts-admin-travel-service  log.error_count     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
3       ts-admin-travel-service    log.row_count     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
7          ts-assurance-service    log.row_count     25     1706145600  2024-01-25T01:20:00Z     1706147040  2024-01-25T01:44:00Z
..                          ...              ...    ...            ...                   ...            ...                   ...
23           ts-consign-service    log.row_count     24     1706145600  2024-01-25T01:20:00Z     1706146980  2024-01-25T01:43:00Z
21       ts-consign-price-mongo    log.row_count     24     1706145600  2024-01-25T01:20:00Z     1706146980  2024-01-25T01:43:00Z
20       ts-consign-price-mongo  log.error_count     24     1706145600  2024-01-25T01:20:00Z     1706146980  2024-01-25T01:43:00Z
63             ts-station-mongo    log.row_count     24     1706145600  2024-01-25T01:20:00Z     1706146980  2024-01-25T01:43:00Z
62             ts-station-mongo  log.error_count     24     1706145600  2024-01-25T01:20:00Z     1706146980  2024-01-25T01:43:00Z

[50 rows x 7 columns],    total_rows earliest_timestamp earliest_timestamp_dt latest_timestamp latest_timestamp_dt
0           0               <NA>                  None             <NA>                None)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# 1) Load metric.csv and parse timestamps as UTC datetimes (reuse variable names)
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure we have the global P95 table in metric_grp (computed earlier). If not present, compute it.
try:
    metric_grp
except NameError:
    metric_grp = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(p95_value=('value', lambda s: s.quantile(0.95)))
        .reset_index()
    )

# 2) Filter metric.csv to the incident window (inclusive)
start = '2024-01-25T01:17:04Z'
end =   '2024-01-25T01:47:04Z'
start_ts = int(pd.to_datetime(start, utc=True).timestamp())
end_ts = int(pd.to_datetime(end, utc=True).timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# 3) Join the filtered points with the global P95 table on (cmdb_id, kpi_name)
metric_window = metric_window.merge(
    metric_grp[['cmdb_id', 'kpi_name', 'p95_value']],
    on=['cmdb_id', 'kpi_name'],
    how='left',
    validate='m:1'
)

# 4) Compute per-group anomaly summaries
def longest_consecutive_run(timestamps):
    # timestamps: sorted numeric seconds of anomaly points
    if len(timestamps) == 0:
        return 0
    if len(timestamps) == 1:
        return 1
    diffs = np.diff(np.array(timestamps, dtype=np.int64))
    # consecutive minutes are diffs == 60
    # group by breaks where diff != 60
    runs = []
    run_len = 1
    for d in diffs:
        if d == 60:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return int(max(runs)) if runs else int(run_len)

groups = []
grp_obj = metric_window.groupby(['cmdb_id', 'kpi_name'], sort=False)

for (cmdb_id, kpi_name), g in grp_obj:
    pts = g.shape[0]
    # drop rows where p95_value is NaN (no global p95) -> treat as no-anomaly potential
    p95 = g['p95_value'].iloc[0]
    # anomaly condition: value >= p95_value (if p95 is NaN, no anomalies)
    if pd.isna(p95):
        anomalies = g.iloc[0:0]  # empty
    else:
        anomalies = g[g['value'] >= p95].sort_values('timestamp')
    anom_count = anomalies.shape[0]
    if anom_count > 0:
        earliest_anom_ts = int(anomalies['timestamp'].iloc[0])
        earliest_anom_dt = pd.to_datetime(earliest_anom_ts, unit='s', utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
        anom_timestamps = anomalies['timestamp'].astype('int64').tolist()
        longest_run = longest_consecutive_run(anom_timestamps)
        sample_anom_ts = [int(x) for x in anom_timestamps[:5]]
    else:
        earliest_anom_ts = pd.NA
        earliest_anom_dt = None
        longest_run = 0
        sample_anom_ts = []
    max_value = g['value'].max() if pts > 0 else pd.NA
    # compute max_over_p95
    if pd.isna(p95):
        max_over_p95 = pd.NA
    else:
        if p95 == 0:
            max_over_p95 = float('inf') if (not pd.isna(max_value)) else pd.NA
        else:
            max_over_p95 = float(max_value) / float(p95) if not pd.isna(max_value) else pd.NA

    groups.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'points_in_window': int(pts),
        'anomaly_count': int(anom_count),
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'earliest_anomaly_dt': earliest_anom_dt,
        'longest_consecutive_anomaly': int(longest_run),
        'max_value_in_window': float(max_value) if not pd.isna(max_value) else pd.NA,
        'p95_value': float(p95) if not pd.isna(p95) else pd.NA,
        'max_over_p95': max_over_p95,
        'sample_anomalous_timestamps': sample_anom_ts
    })

anomalies_summary = pd.DataFrame(groups)

# 5) Return top 20 groups sorted by anomaly_count desc then longest_consecutive_anomaly desc
anomalies_summary = anomalies_summary.sort_values(
    ['anomaly_count', 'longest_consecutive_anomaly'],
    ascending=[False, False]
).reset_index(drop=True)

# Keep only requested columns and limit to 20 rows
anomalies_summary = anomalies_summary.loc[:, [
    'cmdb_id', 'kpi_name', 'points_in_window', 'anomaly_count',
    'earliest_anomaly_timestamp', 'earliest_anomaly_dt',
    'longest_consecutive_anomaly', 'max_value_in_window',
    'p95_value', 'max_over_p95', 'sample_anomalous_timestamps'
]].head(20)

anomalies_summary

```
Out[2]:
```
Summary of findings (plain English):

- Almost all top groups (top 20 returned) show every point in the incident window flagged as anomalous: points_in_window = 25 and anomaly_count = 25, i.e., 25/25 minutes were >= the global P95.
- The earliest anomaly for these groups occurs at 1706145600 → 2024-01-25T01:20:00Z, and the anomalies form a continuous run: longest_consecutive_anomaly = 25 minutes for these groups.
- Several KPIs have p95_value == 0 (e.g., many "error" metrics and some disk/memory metrics). For those, max_over_p95 is reported as inf because dividing by zero yields infinite ratio — this indicates the raw P95 baseline is zero and any positive observed value exceeds it.
- Some groups have non-zero P95 and show max_over_p95 ≈ 1.0 (for example, socket KPIs with p95 3 or 4 and max_value equal to that p95), meaning values reached about the P95 but did not far exceed it.
- Example notable entries: 
  - ts-admin-basic-info-service / error — 25/25 anomalies, p95 = 0 (inf ratio), earliest 2024-01-25T01:20:00Z.
  - ts-assurance-mongo / socket — 25/25 anomalies, p95 = 3.0, max = 3.0, max_over_p95 = 1.0.
  - ts-avatar-service / socket — 25/25 anomalies, p95 = 5.0, max = 5.0, max_over_p95 = 1.0.
  - ts-food-mysql / mem — 25/25 anomalies, p95 ~= 5.9981e8, max equals p95, ratio = 1.0.
- Interpretation: For the inspected incident window, many metrics across multiple services were at-or-above their global P95 for the entire window, starting at ~01:20 UTC. Metrics with p95 == 0 (especially error counts) should be investigated first because any positive value will appear as an extreme breach of the global P95 baseline.
- Recommended next steps: inspect services with high error metrics and p95==0, check logs/traces for those services around 01:20–01:44 UTC, and verify whether the P95 baselines are meaningful (zero baselines may require re-evaluation).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  points_in_window  anomaly_count  earliest_anomaly_timestamp   earliest_anomaly_dt  longest_consecutive_anomaly  max_value_in_window     p95_value  max_over_p95                        sample_anomalous_timestamps
0   ts-admin-basic-info-service       error                25             25                  1706145600  2024-01-25T01:20:00Z                           25         3.341667e-02  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
1       ts-admin-travel-service       error                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
2            ts-assurance-mongo      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         3.000000e+00  3.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
3          ts-assurance-service       error                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
4             ts-avatar-service      diskio                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
5             ts-avatar-service      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         5.000000e+00  5.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
6              ts-basic-service       error                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
7               ts-config-mongo      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         4.000000e+00  4.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
8              ts-consign-mongo      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         4.000000e+00  4.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
9        ts-consign-price-mongo      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         4.000000e+00  4.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
10           ts-consign-service       error                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
11            ts-contacts-mongo      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         3.000000e+00  3.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
12          ts-contacts-service    workload                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
13            ts-food-map-mongo      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         5.000000e+00  5.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
14                ts-food-mongo      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         3.000000e+00  3.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
15                ts-food-mysql      diskio                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
16                ts-food-mysql         mem                25             25                  1706145600  2024-01-25T01:20:00Z                           25         5.998100e+08  5.998100e+08           1.0  [1706145600, 1706145660, 1706145720, 170614578...
17                ts-food-mysql      socket                25             25                  1706145600  2024-01-25T01:20:00Z                           25         2.000000e+00  2.000000e+00           1.0  [1706145600, 1706145660, 1706145720, 170614578...
18              ts-food-service       error                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00           inf  [1706145600, 1706145660, 1706145720, 170614578...
19    ts-inside-payment-service  latency-50                25             25                  1706145600  2024-01-25T01:20:00Z                           25         1.750000e-02  1.750000e-02           1.0  [1706145600, 1706145660, 1706145720, 170614578...```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp columns exist and parsed
    if 'timestamp_dt' not in metric_df.columns:
        metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
        metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Compute global P95 per (cmdb_id, kpi_name) for candidate services using full series
metric_candidates_full = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()
if metric_candidates_full.shape[0] == 0:
    # No metric rows for candidates at all -> explicit empty result
    result = pd.DataFrame([{
        'message': 'No metric data found for candidate services.'
    }])
else:
    metric_p95 = (
        metric_candidates_full
        .groupby(['cmdb_id', 'kpi_name'], dropna=False, sort=False)
        .agg(p95_value=('value', lambda s: float(np.nan) if s.dropna().shape[0]==0 else float(s.quantile(0.95))))
        .reset_index()
    )

    # 2) Filter metric points to the incident window (inclusive)
    start = '2024-01-25T01:17:04Z'
    end =   '2024-01-25T01:47:04Z'
    start_ts = int(pd.to_datetime(start, utc=True).timestamp())
    end_ts = int(pd.to_datetime(end, utc=True).timestamp())

    metric_window = metric_candidates_full[
        (metric_candidates_full['timestamp'] >= start_ts) &
        (metric_candidates_full['timestamp'] <= end_ts)
    ].copy()

    # 3) Join filtered points with global P95 table
    metric_window = metric_window.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left', validate='m:1')

    # Helper: longest consecutive run (in minutes) given sorted numeric seconds
    def longest_consecutive_run(ts_list):
        if len(ts_list) == 0:
            return 0
        arr = np.array(ts_list, dtype=np.int64)
        if arr.size == 1:
            return 1
        diffs = np.diff(arr)
        runs = []
        run_len = 1
        for d in diffs:
            if d == 60:
                run_len += 1
            else:
                runs.append(run_len)
                run_len = 1
        runs.append(run_len)
        return int(max(runs)) if runs else int(run_len)

    # 4) For each (cmdb_id, kpi_name) compute the requested summary
    groups = []
    grp_obj = metric_window.groupby(['cmdb_id', 'kpi_name'], sort=False)

    for (cmdb_id, kpi_name), g in grp_obj:
        pts = int(g.shape[0])
        p95 = g['p95_value'].iloc[0]  # may be NaN
        # Determine anomalies: value >= p95_value (only if p95 is a number)
        if pd.isna(p95):
            anomalies = g.iloc[0:0]
        else:
            anomalies = g[g['value'] >= p95].sort_values('timestamp')
        anom_count = int(anomalies.shape[0])
        if anom_count > 0:
            earliest_anom_ts = int(anomalies['timestamp'].iloc[0])
            earliest_anom_dt = pd.to_datetime(earliest_anom_ts, unit='s', utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
            anom_ts_list = anomalies['timestamp'].astype('int64').tolist()
            longest_run = longest_consecutive_run(anom_ts_list)
            sample_anom_ts = [int(x) for x in anom_ts_list[:5]]
        else:
            earliest_anom_ts = pd.NA
            earliest_anom_dt = None
            longest_run = 0
            sample_anom_ts = []

        max_value = float(g['value'].max()) if pts > 0 else pd.NA

        # compute max_over_p95 (use 'inf' string if p95 == 0 and max_value > 0)
        if pd.isna(p95):
            max_over_p95 = pd.NA
        else:
            if p95 == 0:
                if not (pd.isna(max_value)) and max_value > 0:
                    max_over_p95 = 'inf'
                else:
                    # both zero => 0/0 treated as NaN -> use pd.NA
                    max_over_p95 = pd.NA
            else:
                max_over_p95 = float(max_value) / float(p95)

        groups.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'points_in_window': pts,
            'anomaly_count': anom_count,
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'earliest_anomaly_dt': earliest_anom_dt,
            'longest_consecutive_anomaly': int(longest_run),
            'max_value_in_window': max_value,
            'p95_value': float(p95) if not pd.isna(p95) else pd.NA,
            'max_over_p95': max_over_p95,
            'sample_anomalous_timestamps': sample_anom_ts
        })

    summary_df = pd.DataFrame(groups)

    # If some candidate groups had no points in the window, include them with zeros?
    # The user asked "for each (cmdb_id, kpi_name) pair among those services" -> ensure we include groups present in metric_p95 even if they have 0 points in window
    # Add missing groups (present in metric_p95 but not in summary_df) with points_in_window=0
    existing_pairs = set((r['cmdb_id'], r['kpi_name']) for r in groups)
    missing_rows = []
    for _, row in metric_p95.iterrows():
        key = (row['cmdb_id'], row['kpi_name'])
        if key not in existing_pairs:
            missing_rows.append({
                'cmdb_id': row['cmdb_id'],
                'kpi_name': row['kpi_name'],
                'points_in_window': 0,
                'anomaly_count': 0,
                'earliest_anomaly_timestamp': pd.NA,
                'earliest_anomaly_dt': None,
                'longest_consecutive_anomaly': 0,
                'max_value_in_window': pd.NA,
                'p95_value': float(row['p95_value']) if not pd.isna(row['p95_value']) else pd.NA,
                'max_over_p95': pd.NA,
                'sample_anomalous_timestamps': []
            })
    if missing_rows:
        summary_df = pd.concat([summary_df, pd.DataFrame(missing_rows)], ignore_index=True, sort=False)

    # 5) Sort and limit to 20 rows
    summary_df = summary_df.sort_values(['anomaly_count', 'longest_consecutive_anomaly'], ascending=[False, False]).reset_index(drop=True)

    # Keep requested columns and top 20
    result_df = summary_df.loc[:, [
        'cmdb_id', 'kpi_name', 'points_in_window', 'anomaly_count',
        'earliest_anomaly_timestamp', 'earliest_anomaly_dt',
        'longest_consecutive_anomaly', 'max_value_in_window',
        'p95_value', 'max_over_p95', 'sample_anomalous_timestamps'
    ]].head(20)

    # If there are no anomalies across all candidate groups, return explicit empty result indication
    if result_df['anomaly_count'].sum() == 0:
        result = pd.DataFrame([{
            'message': 'No anomalies found for the specified candidate services in the incident window.'
        }])
    else:
        result = result_df

# Display the result (variable)
result

```
Out[3]:
```
Summary (plain English):

- Overall: Among the five candidate services, anomalies were detected in multiple KPI groups during the incident window, with varying severity and duration.

- Most notable:
  - ts-travel-service / error: every point in the window was anomalous (points_in_window=25, anomaly_count=25). Earliest anomaly at 1706145600 → 2024-01-25T01:20:00Z, longest consecutive run = 25 minutes. p95 for this error metric is 0 (baseline zero), so ratios are not meaningful — any positive error counts should be investigated first.
  
- Short spikes (1–2 minute runs) across services:
  - ts-auth-service:
    - latency-50: 2 anomaly points, earliest 1706146920 → 2024-01-25T01:42:00Z, longest run = 2, max_over_p95 ≈ 1.061 (small exceedance).
    - mem: 2 anomalies, earliest 1706146980 → 2024-01-25T01:43:00Z, max_over_p95 ≈ 1.00008 (essentially at P95).
    - cpu, diskio, latency-90, socket, workload: also show small numbers of anomalies (mostly 1–2 points), with modest exceedances (ratios ~1.04–1.12) except cpu/diskio spikes which are larger for some groups.
  - ts-order-service:
    - workload, cpu, diskio: anomaly_count 2 for each; cpu shows the largest relative exceedance here (max_over_p95 ≈ 2.56), earliest cpu anomaly at 1706146500 → 2024-01-25T01:35:00Z.
  - ts-route-service:
    - cpu, latency-50, latency-90, mem: anomaly_count 2, earliest at 1706146980 → 2024-01-25T01:43:00Z; cpu max_over_p95 ≈ 1.92 (notable exceedance).
  - ts-travel-service (other KPIs):
    - latency-50, latency-90, mem, socket: each had 2 anomalous points (earliest between 01:34 and 01:38 UTC), with modest exceedances (ratios ~1.01–1.38).

- p95==0 cases: Several "error" metrics have p95_value = 0 (including ts-travel-service error and some route/error groups). That produces undefined/infinite ratios for max_over_p95; these should be prioritized because any non-zero error value is above the zero baseline.

- Interpretation & recommended next steps:
  1. Immediately investigate ts-travel-service error metric (continuous anomalies starting ~01:20 UTC) — check service logs/traces and error stack traces around 01:20–01:44 UTC.
  2. Investigate the short spikes in ts-auth, ts-order, and ts-route (notably cpu on ts-order and cpu on ts-route with larger ratios) around 01:34–01:43 UTC; check traces and request/latency patterns for these timestamps (sample anomalous timestamps are provided for each group).
  3. Re-evaluate zero p95 baselines for error metrics (zero baseline makes ratio-based severity misleading); treat any positive error counts as high-priority alerts.
  4. Correlate these metric anomalies with logs and traces for the listed timestamps to find root cause(s).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  points_in_window  anomaly_count  earliest_anomaly_timestamp   earliest_anomaly_dt  longest_consecutive_anomaly  max_value_in_window     p95_value max_over_p95                        sample_anomalous_timestamps
0   ts-travel-service       error                25             25                  1706145600  2024-01-25T01:20:00Z                           25         0.000000e+00  0.000000e+00         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
1     ts-auth-service  latency-50                25              2                  1706146920  2024-01-25T01:42:00Z                            2         2.784840e-01  2.625091e-01     1.060854                           [1706146920, 1706146980]
2     ts-auth-service         mem                25              2                  1706146980  2024-01-25T01:43:00Z                            2         2.420314e+08  2.420134e+08     1.000075                           [1706146980, 1706147040]
3    ts-order-service    workload                25              2                  1706146980  2024-01-25T01:43:00Z                            2         6.670000e-01  5.651033e-01     1.180315                           [1706146980, 1706147040]
4    ts-route-service         cpu                25              2                  1706146980  2024-01-25T01:43:00Z                            2         7.685398e+00  3.996382e+00     1.923089                           [1706146980, 1706147040]
5    ts-route-service  latency-50                25              2                  1706146980  2024-01-25T01:43:00Z                            2         4.500000e-03  3.747696e-03     1.200738                           [1706146980, 1706147040]
6    ts-route-service  latency-90                25              2                  1706146980  2024-01-25T01:43:00Z                            2         9.660526e-03  8.318437e-03     1.161339                           [1706146980, 1706147040]
7    ts-route-service         mem                25              2                  1706146980  2024-01-25T01:43:00Z                            2         2.605641e+08  2.592325e+08     1.005137                           [1706146980, 1706147040]
8   ts-travel-service  latency-50                25              2                  1706146440  2024-01-25T01:34:00Z                            2         3.514459e-01  2.554293e-01     1.375903                           [1706146440, 1706146500]
9   ts-travel-service  latency-90                25              2                  1706146440  2024-01-25T01:34:00Z                            2         8.041150e-01  7.331927e-01     1.096731                           [1706146440, 1706146500]
10  ts-travel-service         mem                25              2                  1706146680  2024-01-25T01:38:00Z                            2         2.785879e+08  2.785223e+08     1.000235                           [1706146680, 1706146740]
11  ts-travel-service      socket                25              2                  1706146620  2024-01-25T01:37:00Z                            2         2.860000e+01  2.842667e+01     1.006098                           [1706146620, 1706146680]
12   ts-route-service       error                 2              2                  1706146980  2024-01-25T01:43:00Z                            2         0.000000e+00  0.000000e+00         <NA>                           [1706146980, 1706147040]
13    ts-auth-service         cpu                25              2                  1706146320  2024-01-25T01:32:00Z                            1         1.638986e+01  1.556116e+01     1.053254                           [1706146320, 1706146920]
14    ts-auth-service      diskio                25              2                  1706145960  2024-01-25T01:26:00Z                            1         6.612189e+04  6.074378e+04     1.088538                           [1706145960, 1706146200]
15    ts-auth-service  latency-90                25              2                  1706146860  2024-01-25T01:41:00Z                            1         9.763912e-01  8.703697e-01     1.121812                           [1706146860, 1706146980]
16    ts-auth-service      socket                25              2                  1706146380  2024-01-25T01:33:00Z                            1         1.466667e+01  1.364333e+01     1.075006                           [1706146380, 1706146860]
17    ts-auth-service    workload                25              2                  1706146740  2024-01-25T01:39:00Z                            1         3.333000e+00  3.192033e+00     1.044162                           [1706146740, 1706147040]
18   ts-order-service         cpu                25              2                  1706146500  2024-01-25T01:35:00Z                            1         2.500621e+00  9.768241e-01      2.55995                           [1706146500, 1706146620]
19   ts-order-service      diskio                25              2                  1706145960  2024-01-25T01:26:00Z                            1         6.028957e+04  5.934609e+04     1.015898                           [1706145960, 1706146080]```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import re

# Reuse trace_df if present, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp numeric and parsed as UTC datetimes
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (do NOT filter first)
trace_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False, sort=False)
    .agg(p95_value=('value', lambda s: float(np.nan) if s.dropna().shape[0]==0 else float(s.quantile(0.95))))
    .reset_index()
)

# 2) Filter trace points to the incident window (inclusive)
start = '2024-01-25T01:17:04Z'
end =   '2024-01-25T01:47:04Z'
start_ts = int(pd.to_datetime(start, utc=True).timestamp())
end_ts = int(pd.to_datetime(end, utc=True).timestamp())

trace_window = trace_df[
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# 3) Keep only trace groups where cmdb_id is in candidates OR trace_name contains any candidate service name
# Build regex pattern safely
pattern = "|".join(re.escape(c) for c in candidates)
mask = trace_window['cmdb_id'].isin(candidates) | trace_window['trace_name'].fillna("").str.contains(pattern, regex=True)
trace_window = trace_window[mask].copy()

# If no matching traces, return explicit empty result
if trace_window.shape[0] == 0:
    trace_result = pd.DataFrame([{
        'message': 'No matching trace groups found for the candidate services in the incident window.'
    }])
else:
    # 4) Join filtered points with global P95 per group
    trace_window = trace_window.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left', validate='m:1')

    # Helper to compute longest consecutive run in minutes for sorted list of numeric seconds
    def longest_consecutive_run(ts_list):
        if len(ts_list) == 0:
            return 0
        arr = np.array(ts_list, dtype=np.int64)
        if arr.size == 1:
            return 1
        diffs = np.diff(arr)
        runs = []
        run_len = 1
        for d in diffs:
            if d == 60:
                run_len += 1
            else:
                runs.append(run_len)
                run_len = 1
        runs.append(run_len)
        return int(max(runs)) if runs else int(run_len)

    # Group and summarize
    groups = []
    for (cmdb_id, trace_name), g in trace_window.groupby(['cmdb_id', 'trace_name'], sort=False):
        pts = int(g.shape[0])
        p95 = g['p95_value'].iloc[0]
        if pd.isna(p95):
            anomalies = g.iloc[0:0]
        else:
            anomalies = g[g['value'] >= p95].sort_values('timestamp')
        anom_count = int(anomalies.shape[0])

        if anom_count > 0:
            earliest_anom_ts = int(anomalies['timestamp'].iloc[0])
            earliest_anom_dt = pd.to_datetime(earliest_anom_ts, unit='s', utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
            anom_ts_list = anomalies['timestamp'].astype('int64').tolist()
            longest_run = longest_consecutive_run(anom_ts_list)
            sample_anom_ts = [int(x) for x in anom_ts_list[:5]]
        else:
            earliest_anom_ts = pd.NA
            earliest_anom_dt = None
            longest_run = 0
            sample_anom_ts = []

        max_value = float(g['value'].max()) if pts > 0 else pd.NA

        # compute max_over_p95
        if pd.isna(p95):
            max_over_p95 = pd.NA
        else:
            if p95 == 0:
                if not (pd.isna(max_value)) and max_value > 0:
                    max_over_p95 = 'inf'
                else:
                    max_over_p95 = pd.NA
            else:
                max_over_p95 = float(max_value) / float(p95)

        groups.append({
            'cmdb_id': cmdb_id,
            'trace_name': trace_name,
            'points_in_window': pts,
            'anomaly_count': anom_count,
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'earliest_anomaly_dt': earliest_anom_dt,
            'longest_consecutive_anomaly': int(longest_run),
            'max_value_in_window': max_value,
            'p95_value': float(p95) if not pd.isna(p95) else pd.NA,
            'max_over_p95': max_over_p95,
            'sample_anomalous_timestamps': sample_anom_ts
        })

    trace_summary = pd.DataFrame(groups)

    # 5) Sort and limit to top 20 groups
    trace_result = trace_summary.sort_values(['anomaly_count', 'longest_consecutive_anomaly'], ascending=[False, False]).reset_index(drop=True).head(20)

# Display the compact result
trace_result

```
Out[4]:
```
Summary (plain English):

- What I found: many trace groups that mention the candidate services (including root traces like trace.to_ts-auth-service.error_rate, trace.to_ts-order-service.error_rate, and service-local traces) show every point in the incident window marked as "anomalous" by the current threshold logic: points_in_window = 25 and anomaly_count = 25, with earliest anomaly at 1706145600 → 2024-01-25T01:20:00Z and a continuous 25-minute run.

- Why this is not actionable as-is: those trace groups have p95_value = 0.0 and max_value_in_window = 0.0. Because the rule used was value >= p95, zero baselines cause all zero-valued points to be classified as anomalies (or produce undefined ratios). This makes the anomaly labels meaningless for these trace.error_rate series.

- Useful signals to focus on:
  - Look for trace groups with p95 > 0 and max_value > p95 (non-zero exceedances). In this result the reported max_value_in_window is 0.0 for the top groups, so no true exceedances are present among these error_rate traces.
  - Correlate with the metric-level findings (metrics showed many real error and other KPI anomalies starting ~2024-01-25T01:20:00Z). Prioritize trace groups that show non-zero error/latency values or whose names contain the candidate services and have p95 > 0.

- Recommended next steps:
  1. Re-evaluate/adjust p95 baselines for trace.error_rate series (zero baselines should not be used to flag anomalies via value >= p95).
  2. Inspect traces and logs for candidate services at timestamps around 2024-01-25T01:20:00Z and the later short spikes (01:34–01:44 UTC) identified in metric analysis.
  3. Focus on trace series with p95 > 0 and max_value_in_window > p95 (non-zero exceedances) for real trace-level anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       trace_name  points_in_window  anomaly_count  earliest_anomaly_timestamp   earliest_anomaly_dt  longest_consecutive_anomaly  max_value_in_window  p95_value max_over_p95                        sample_anomalous_timestamps
0                        root              trace.to_ts-auth-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
1                        root             trace.to_ts-order-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
2                        root            trace.to_ts-travel-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
3     ts-admin-travel-service            trace.to_ts-travel-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
4             ts-auth-service                       trace.from_root.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
5             ts-auth-service            trace.from_ts-auth-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
6             ts-auth-service              trace.to_ts-auth-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
7             ts-food-service            trace.to_ts-travel-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
8   ts-inside-payment-service             trace.to_ts-order-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
9            ts-order-service                       trace.from_root.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
10           ts-order-service  trace.from_ts-inside-payment-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
11           ts-order-service           trace.from_ts-order-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
12           ts-order-service             trace.to_ts-order-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
13           ts-order-service           trace.to_ts-station-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
14           ts-route-service           trace.from_ts-route-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
15           ts-route-service          trace.from_ts-travel-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
16           ts-route-service         trace.from_ts-travel2-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
17           ts-route-service             trace.to_ts-route-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
18         ts-station-service           trace.from_ts-order-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
19      ts-ticketinfo-service          trace.from_ts-travel-service.error_rate                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse log_df if present, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Ensure timestamp numeric and parsed as UTC datetimes
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Compute global P95 per (cmdb_id, log_name) for candidate services using full series (do NOT filter)
logs_candidates_full = log_df[log_df['cmdb_id'].isin(candidates)].copy()

if logs_candidates_full.shape[0] == 0:
    # No log rows for candidates at all -> explicit empty result
    result_logs = pd.DataFrame([{'message': 'No log data found for candidate services.'}])
else:
    log_p95 = (
        logs_candidates_full
        .groupby(['cmdb_id', 'log_name'], dropna=False, sort=False)
        .agg(p95_value=('value', lambda s: float(np.nan) if s.dropna().shape[0]==0 else float(s.quantile(0.95))))
        .reset_index()
    )

    # 2) Filter log points to the incident window (inclusive)
    start = '2024-01-25T01:17:04Z'
    end =   '2024-01-25T01:47:04Z'
    start_ts = int(pd.to_datetime(start, utc=True).timestamp())
    end_ts = int(pd.to_datetime(end, utc=True).timestamp())

    logs_window = logs_candidates_full[
        (logs_candidates_full['timestamp'] >= start_ts) &
        (logs_candidates_full['timestamp'] <= end_ts)
    ].copy()

    # 3) Join filtered points with global P95 table
    logs_window = logs_window.merge(log_p95, on=['cmdb_id', 'log_name'], how='left', validate='m:1')

    # Helper for longest consecutive run (minutes) given sorted numeric seconds
    def longest_consecutive_run(ts_list):
        if len(ts_list) == 0:
            return 0
        arr = np.array(ts_list, dtype=np.int64)
        if arr.size == 1:
            return 1
        diffs = np.diff(arr)
        runs = []
        run_len = 1
        for d in diffs:
            if d == 60:
                run_len += 1
            else:
                runs.append(run_len)
                run_len = 1
        runs.append(run_len)
        return int(max(runs)) if runs else int(run_len)

    # 4) For each (cmdb_id, log_name) compute requested summary
    groups = []
    grp_obj = logs_window.groupby(['cmdb_id', 'log_name'], sort=False)

    for (cmdb_id, log_name), g in grp_obj:
        pts = int(g.shape[0])
        p95 = g['p95_value'].iloc[0]
        # anomaly condition: value >= p95_value (only if p95 is number)
        if pd.isna(p95):
            anomalies = g.iloc[0:0]
        else:
            anomalies = g[g['value'] >= p95].sort_values('timestamp')
        anom_count = int(anomalies.shape[0])

        if anom_count > 0:
            earliest_anom_ts = int(anomalies['timestamp'].iloc[0])
            earliest_anom_dt = pd.to_datetime(earliest_anom_ts, unit='s', utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
            anom_ts_list = anomalies['timestamp'].astype('int64').tolist()
            longest_run = longest_consecutive_run(anom_ts_list)
            sample_anom_ts = [int(x) for x in anom_ts_list[:5]]
        else:
            earliest_anom_ts = pd.NA
            earliest_anom_dt = None
            longest_run = 0
            sample_anom_ts = []

        max_value = float(g['value'].max()) if pts > 0 else pd.NA

        # compute max_over_p95: 'inf' if p95==0 and max_value>0
        if pd.isna(p95):
            max_over_p95 = pd.NA
        else:
            if p95 == 0:
                if not (pd.isna(max_value)) and max_value > 0:
                    max_over_p95 = 'inf'
                else:
                    max_over_p95 = pd.NA
            else:
                max_over_p95 = float(max_value) / float(p95)

        groups.append({
            'cmdb_id': cmdb_id,
            'log_name': log_name,
            'points_in_window': pts,
            'anomaly_count': anom_count,
            'earliest_anomaly_timestamp': earliest_anom_ts,
            'earliest_anomaly_dt': earliest_anom_dt,
            'longest_consecutive_anomaly': int(longest_run),
            'max_value_in_window': max_value,
            'p95_value': float(p95) if not pd.isna(p95) else pd.NA,
            'max_over_p95': max_over_p95,
            'sample_anomalous_timestamps': sample_anom_ts
        })

    summary_logs = pd.DataFrame(groups)

    # Include groups present in log_p95 but with zero points in window
    existing_pairs = set((r['cmdb_id'], r['log_name']) for r in groups)
    missing_rows = []
    for _, row in log_p95.iterrows():
        key = (row['cmdb_id'], row['log_name'])
        if key not in existing_pairs:
            missing_rows.append({
                'cmdb_id': row['cmdb_id'],
                'log_name': row['log_name'],
                'points_in_window': 0,
                'anomaly_count': 0,
                'earliest_anomaly_timestamp': pd.NA,
                'earliest_anomaly_dt': None,
                'longest_consecutive_anomaly': 0,
                'max_value_in_window': pd.NA,
                'p95_value': float(row['p95_value']) if not pd.isna(row['p95_value']) else pd.NA,
                'max_over_p95': pd.NA,
                'sample_anomalous_timestamps': []
            })
    if missing_rows:
        summary_logs = pd.concat([summary_logs, pd.DataFrame(missing_rows)], ignore_index=True, sort=False)

    # 5) Sort and limit to 20 rows
    result_df = summary_logs.sort_values(['anomaly_count', 'longest_consecutive_anomaly'], ascending=[False, False]).reset_index(drop=True)
    result_df = result_df.loc[:, [
        'cmdb_id', 'log_name', 'points_in_window', 'anomaly_count',
        'earliest_anomaly_timestamp', 'earliest_anomaly_dt',
        'longest_consecutive_anomaly', 'max_value_in_window',
        'p95_value', 'max_over_p95', 'sample_anomalous_timestamps'
    ]].head(20)

    # If no anomalies across all candidate groups, return explicit empty result indicator
    if result_df['anomaly_count'].sum() == 0:
        result_logs = pd.DataFrame([{'message': 'No anomalies found for the specified candidate services in the incident window.'}])
    else:
        result_logs = result_df

# Display result_logs (compact)
result_logs

```
Out[5]:
```
Summary (plain English):

- What’s noisy but not actionable:
  - For log.error_count in ts-auth-service, ts-order-service, ts-route-service, and ts-travel-service the algorithm flagged every minute in the window as "anomalous" (25/25) starting at 1706145600 → 2024-01-25T01:20:00Z with a 25-minute continuous run.
  - These detections are spurious because the global P95 for those error_count series is 0.0 and the observed max in-window is also 0.0. The rule value >= p95 therefore classifies zeros as anomalies. Treat these as false positives until baselines/logic are adjusted.

- Real signals to investigate:
  - log.row_count spikes (real exceedances of P95):
    - ts-route-service / log.row_count: 2 anomaly points, earliest 1706146920 → 2024-01-25T01:42:00Z; max_in_window = 525 vs p95 = 378.6 (max_over_p95 ≈ 1.39). Sample anomalous timestamps: [1706146920, 1706146980].
    - ts-travel-service / log.row_count: 2 anomaly points, earliest 1706146860 → 2024-01-25T01:41:00Z; max = 603 vs p95 = 555.4 (ratio ≈ 1.09). Sample: [1706146860, 1706146920].
    - ts-auth-service / log.row_count: 2 anomaly points, earliest 1706146020 → 2024-01-25T01:27:00Z; max = 390 vs p95 = 376.8 (ratio ≈ 1.04). Sample: [1706146020, 1706146740].
    - ts-order-service / log.row_count: 2 anomaly points, earliest 1706145900 → 2024-01-25T01:25:00Z; max = 59 vs p95 = 58.8 (ratio ≈ 1.003). Sample: [1706145900, 1706146500].

- Interpretation & next steps:
  1. Ignore the error_count "anomalies" driven by zero P95 (false positives). Re-evaluate thresholds for error_count metrics (or require value > p95 rather than >=, and handle p95==0 specially).
  2. Prioritize investigation of log.row_count spikes in ts-route-service and ts-travel-service (largest relative exceedances), then ts-auth and ts-order. Check logs and request/ingestion rates around the sample timestamps (01:25–01:42 UTC).
  3. Correlate these row_count spikes with trace and metric anomalies previously identified to find root cause(s).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  points_in_window  anomaly_count  earliest_anomaly_timestamp   earliest_anomaly_dt  longest_consecutive_anomaly  max_value_in_window  p95_value max_over_p95                        sample_anomalous_timestamps
0    ts-auth-service  log.error_count                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
1   ts-order-service  log.error_count                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
2   ts-route-service  log.error_count                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
3  ts-travel-service  log.error_count                25             25                  1706145600  2024-01-25T01:20:00Z                           25                  0.0        0.0         <NA>  [1706145600, 1706145660, 1706145720, 170614578...
4   ts-route-service    log.row_count                25              2                  1706146920  2024-01-25T01:42:00Z                            2                525.0      378.6     1.386688                           [1706146920, 1706146980]
5  ts-travel-service    log.row_count                25              2                  1706146860  2024-01-25T01:41:00Z                            2                603.0      555.4     1.085704                           [1706146860, 1706146920]
6    ts-auth-service    log.row_count                25              2                  1706146020  2024-01-25T01:27:00Z                            1                390.0      376.8     1.035032                           [1706146020, 1706146740]
7   ts-order-service    log.row_count                25              2                  1706145900  2024-01-25T01:25:00Z                            1                 59.0       58.8     1.003401                           [1706145900, 1706146500]```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse metric_df if present, otherwise load and parse timestamps as UTC
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
# Ensure timestamp numeric and datetime parsed (UTC)
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Parameters
service = 'ts-travel-service'
start = '2024-01-25T01:17:04Z'
end =   '2024-01-25T01:47:04Z'
start_ts = int(pd.to_datetime(start, utc=True).timestamp())
end_ts = int(pd.to_datetime(end, utc=True).timestamp())

# 1) Compute global P95 per (cmdb_id, kpi_name) for ts-travel-service using full series (global threshold)
metric_ts_full = metric_df[metric_df['cmdb_id'] == service].copy()
# If no rows at all for service, return explicit empty result
if metric_ts_full.shape[0] == 0:
    result = pd.DataFrame([{'message': f'No metric data found for {service}.'}])
else:
    metric_p95_ts = (
        metric_ts_full
        .groupby(['cmdb_id', 'kpi_name'], dropna=False, sort=False)
        .agg(p95_value=('value', lambda s: float(np.nan) if s.dropna().shape[0]==0 else float(s.quantile(0.95))))
        .reset_index()
    )

    # 2) Filter metric points to incident window (inclusive) for that service
    metric_window = metric_ts_full[
        (metric_ts_full['timestamp'] >= start_ts) &
        (metric_ts_full['timestamp'] <= end_ts)
    ].copy()

    # Helper: longest consecutive run in minutes for sorted numeric seconds
    def longest_consecutive_run(ts_list):
        if len(ts_list) == 0:
            return 0
        arr = np.array(ts_list, dtype=np.int64)
        if arr.size == 1:
            return 1
        diffs = np.diff(arr)
        runs = []
        run_len = 1
        for d in diffs:
            if d == 60:
                run_len += 1
            else:
                runs.append(run_len)
                run_len = 1
        runs.append(run_len)
        return int(max(runs)) if runs else int(run_len)

    # 3) For each kpi_name produce summary. Start from metric_p95_ts to include KPIs with zero points in-window
    groups = []
    for _, row in metric_p95_ts.iterrows():
        kpi = row['kpi_name']
        p95 = row['p95_value']
        g = metric_window[metric_window['kpi_name'] == kpi].copy()
        pts = int(g.shape[0])
        # non-zero points: value > 0
        nonzero = g[g['value'] > 0].sort_values('timestamp')
        nonzero_count = int(nonzero.shape[0])
        if nonzero_count > 0:
            earliest_nonzero_ts = int(nonzero['timestamp'].iloc[0])
            earliest_nonzero_dt = pd.to_datetime(earliest_nonzero_ts, unit='s', utc=True).strftime('%Y-%m-%dT%H:%M:%SZ')
            nonzero_ts_list = nonzero['timestamp'].astype('int64').tolist()
            longest_run = longest_consecutive_run(nonzero_ts_list)
            sample_nonzero_ts = [int(x) for x in nonzero_ts_list[:10]]
        else:
            earliest_nonzero_ts = pd.NA
            earliest_nonzero_dt = None
            longest_run = 0
            sample_nonzero_ts = []

        max_value = float(g['value'].max()) if pts > 0 else pd.NA

        # compute max_over_p95: 'inf' if p95==0 and max_value>0
        if pd.isna(p95):
            max_over_p95 = pd.NA
        else:
            if p95 == 0:
                if not (pd.isna(max_value)) and max_value > 0:
                    max_over_p95 = 'inf'
                else:
                    max_over_p95 = pd.NA
            else:
                max_over_p95 = float(max_value) / float(p95)

        groups.append({
            'kpi_name': kpi,
            'points_in_window': pts,
            'nonzero_count': nonzero_count,
            'earliest_nonzero_timestamp': earliest_nonzero_ts,
            'earliest_nonzero_dt': earliest_nonzero_dt,
            'longest_consecutive_nonzero_run': int(longest_run),
            'max_value_in_window': max_value,
            'p95_value': float(p95) if not pd.isna(p95) else pd.NA,
            'max_over_p95': max_over_p95,
            'sample_nonzero_timestamps': sample_nonzero_ts
        })

    summary_df = pd.DataFrame(groups)

    # If there are no non-zero points across all KPIs in-window, return explicit empty-result indicator
    if summary_df['nonzero_count'].sum() == 0:
        result = pd.DataFrame([{'message': f'No non-zero metric points for {service} in the incident window.'}])
    else:
        # Sort by nonzero_count desc then longest_consecutive_nonzero_run desc and limit to 20 rows
        result = summary_df.sort_values(['nonzero_count', 'longest_consecutive_nonzero_run'], ascending=[False, False]).reset_index(drop=True)
        result = result.loc[:, [
            'kpi_name', 'points_in_window', 'nonzero_count',
            'earliest_nonzero_timestamp', 'earliest_nonzero_dt',
            'longest_consecutive_nonzero_run', 'max_value_in_window',
            'p95_value', 'max_over_p95', 'sample_nonzero_timestamps'
        ]].head(20)

# Display result variable
result

```
Out[6]:
```
Summary (plain English):

- What happened for ts-travel-service during the incident window (2024-01-25T01:17:04Z — 2024-01-25T01:47:04Z):
  - Most KPIs were non-zero for every minute in the window (points_in_window = 25, nonzero_count = 25) and began at 1706145600 → 2024-01-25T01:20:00Z with a continuous 25-minute run.
  - KPIs and notable ratios (max_value_in_window / global P95):
    - diskio: max = 4.888455e+06, p95 = 4.756960e+05, ratio ≈ 10.28 — large disk I/O spike (highest severity).
    - latency-50: max = 0.3514459, p95 = 0.2554293, ratio ≈ 1.376 — meaningful latency increase.
    - latency-90: max = 0.8041150, p95 = 0.7331927, ratio ≈ 1.097 — modest latency increase.
    - cpu: max = 39.35607, p95 = 38.83768, ratio ≈ 1.013 — small CPU rise.
    - mem: max ≈ 2.785879e+08, p95 ≈ 2.785223e+08, ratio ≈ 1.00024 — essentially at the baseline.
    - socket, workload: small exceedances (ratios ≈ 1.006 and 1.021).
  - error metric: all zeros in-window (nonzero_count = 0; p95 = 0), so no non-zero errors observed.

- Interpretation and recommended next steps:
  1. Prioritize investigating the disk I/O spike for ts-travel-service (very large ratio ≈ 10× P95) — check storage, DB calls, bulk reads/writes, and I/O-heavy operations around 01:20–01:44 UTC.
  2. Also examine latency-50 and latency-90 increases (started ~01:20 UTC) and correlate with the disk I/O spike; elevated latencies may be caused by I/O contention.
  3. CPU/memory/socket/workload show minor or baseline-level changes but should be checked if correlated with disk I/O (e.g., processes waiting on I/O).
  4. No application errors were recorded in the metric error series during the window (error remained zero); focus on resource (disk/IO) and performance traces/logs first.
  5. Use trace and log timestamps (earliest non-zero = 2024-01-25T01:20:00Z and sample timestamps provided) to dig into slow traces, DB queries, and any large batch jobs or backups that might explain the disk I/O surge.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  points_in_window  nonzero_count earliest_nonzero_timestamp   earliest_nonzero_dt  longest_consecutive_nonzero_run  max_value_in_window     p95_value max_over_p95                          sample_nonzero_timestamps
0         cpu                25             25                 1706145600  2024-01-25T01:20:00Z                               25         3.935607e+01  3.883768e+01     1.013348  [1706145600, 1706145660, 1706145720, 170614578...
1      diskio                25             25                 1706145600  2024-01-25T01:20:00Z                               25         4.888455e+06  4.756960e+05    10.276427  [1706145600, 1706145660, 1706145720, 170614578...
2  latency-50                25             25                 1706145600  2024-01-25T01:20:00Z                               25         3.514459e-01  2.554293e-01     1.375903  [1706145600, 1706145660, 1706145720, 170614578...
3  latency-90                25             25                 1706145600  2024-01-25T01:20:00Z                               25         8.041150e-01  7.331927e-01     1.096731  [1706145600, 1706145660, 1706145720, 170614578...
4         mem                25             25                 1706145600  2024-01-25T01:20:00Z                               25         2.785879e+08  2.785223e+08     1.000235  [1706145600, 1706145660, 1706145720, 170614578...
5      socket                25             25                 1706145600  2024-01-25T01:20:00Z                               25         2.860000e+01  2.842667e+01     1.006098  [1706145600, 1706145660, 1706145720, 170614578...
6    workload                25             25                 1706145600  2024-01-25T01:20:00Z                               25         1.994383e+00  1.953577e+00     1.020888  [1706145600, 1706145660, 1706145720, 170614578...
7       error                25              0                       <NA>                  None                                0         0.000000e+00  0.000000e+00         <NA>                                                 []```
```